# Sorting Algorithms Introduction

This notebook introduces algorithm complexity, search as a warm-up example, and several sorting algorithms used in DIP321. It favors readable Python implementations that make the algorithmic ideas visible. Python's built-in `sorted` should be preferred in production code.

## Learning goals

By the end of this notebook you should be able to:

- explain the difference between linear, logarithmic, quadratic, factorial, and `n log n` growth;
- identify whether a sorting algorithm works in-place or returns a sorted copy;
- compare bubble sort, selection sort, insertion sort, merge sort, quicksort, heapsort, and counting sort;
- explain why Python uses a hybrid stable sort instead of any one textbook algorithm.

## Sources

Notebook adapted from earlier RTU DIP321 materials and examples inspired by:

- https://github.com/shik3519/programming-concepts-for-data-science
- https://runestone.academy/ns/books/published/pythonds/index.html

In [ ]:
import math
import random
import time
from random import shuffle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

# Algorithm complexity

`Big O notation` describes how an algorithm's resource use grows as input size grows. In this notebook we mostly discuss time complexity.

Common growth rates from fastest to slowest:

| Complexity | Typical meaning |
| --- | --- |
| $O(1)$ | constant work |
| $O(\log n)$ | repeatedly halve the search space |
| $O(n)$ | scan each item once |
| $O(n \log n)$ | divide-and-conquer sorting lower bound for comparison sorts |
| $O(n^2)$ | nested passes over the data |
| $O(n!)$ | try many or all permutations |

Big O is an upper bound. When we know the tight bound, we often use $\Theta(\cdot)$, but in casual algorithm discussions people often say "Big O" when they mean the tight growth rate.

To know more: http://interactivepython.org/runestone/static/pythonds/AlgorithmAnalysis/BigONotation.html

![selection](https://github.com/ValRCS/RTU_Algorithms_DIP321/blob/main/imgs/bigo.png?raw=1)

A useful mental model: constant and logarithmic algorithms scale gently, linear algorithms scale proportionally with input size, quadratic algorithms become expensive quickly, and factorial algorithms are practical only for very small `n`.

Let's understand this through 2 different implementation of search algorithm

## Search algorithm

### Linear search

In [ ]:
def linear_search(sequence, target):
    """Return True if target appears in sequence.

    Worst-case time complexity: O(n), because we may inspect every item.
    """
    for element in sequence:
        if element == target:
            return True
    return False

In [ ]:
l = np.arange(100_000)
l[:5], l[-5:]

In [ ]:
%%timeit
linear_search(l,99_999)

In [ ]:
%%timeit
linear_search(l,1)

Time scales linearly with n. So Big-O is $O(n)$

### Binary search

Iterative algo

In [ ]:
def binarySearchIterative(a, target):
    """Return (found, comparisons) for target in sorted sequence a.

    Time complexity: O(log n). The sequence must already be sorted.
    """
    lower = 0
    upper = len(a) - 1
    comparisons = 0

    while lower <= upper:
        comparisons += 1
        middle = (lower + upper) // 2
        if target == a[middle]:
            return True, comparisons
        if target < a[middle]:
            upper = middle - 1
        else:
            lower = middle + 1

    return False, comparisons

In [ ]:
l = np.arange(100_000)

In [ ]:
binarySearchIterative(l,99_999)

In [ ]:
%%timeit
binarySearchIterative(l,99_999)

In [ ]:
import math
math.log2(100_000)

Time grows logarithmically with `n`, so binary search is $O(\log n)$.

Binary search is much faster than linear search on large sorted inputs because each comparison removes about half of the remaining candidates.

We can do binary search in a recursive way too

In [ ]:
def binarySearchRecursive(a, target, lower=0, upper=None):
    """Recursive binary search without slicing copies."""
    if upper is None:
        upper = len(a) - 1

    if lower > upper:
        return False

    middle = (lower + upper) // 2
    if target == a[middle]:
        return True
    if target < a[middle]:
        return binarySearchRecursive(a, target, lower, middle - 1)
    return binarySearchRecursive(a, target, middle + 1, upper)

In [ ]:
%%timeit
binarySearchRecursive(l,999)

# Sorting algorithms

In [ ]:
# What is the least practical sorting algorithm that you can think of?
# Bogosort: while the data is not sorted, shuffle it and try again.
# https://en.wikipedia.org/wiki/Bogosort

In [ ]:
def is_sorted(data) -> bool:
    """Return True if data is sorted in nondecreasing order."""
    return all(first <= second for first, second in zip(data, data[1:]))


def bogosort(data, max_shuffles=100_000) -> list:
    """Return a sorted copy by repeatedly shuffling.

    This is intentionally terrible: average time is O(n * n!). The guard keeps
    accidental large examples from running forever in Colab.
    """
    result = data.copy()
    shuffles = 0
    while not is_sorted(result):
        if shuffles >= max_shuffles:
            raise RuntimeError(f"bogosort gave up after {max_shuffles:,} shuffles")
        shuffle(result)
        shuffles += 1
    return result


def bogosort_in_place(data, max_shuffles=100_000) -> list:
    """In-place variant for demonstrating mutation."""
    sorted_data = bogosort(data, max_shuffles=max_shuffles)
    data[:] = sorted_data
    return data

In [ ]:
mydata = [1,6,4,3,-7]

In [ ]:
bogodata = bogosort(mydata)
bogodata, mydata  # bogosort returns a sorted copy and leaves original data unchanged

In [ ]:
%%timeit
bogosort(mydata)

In [ ]:
# lets check mydata
mydata

In [ ]:
def bogosort_out_of_place(data) -> list:
    """Backward-compatible name used by older lecture cells."""
    return bogosort(data)

In [ ]:
mydata = [1,6,4,3,-7]
sorted_data = bogosort_out_of_place(mydata)
sorted_data, mydata # original is preserved

In [ ]:
%%timeit
bogosort_out_of_place(mydata)

## Creating Random Numbers

In [ ]:
random.seed(2026)

# Shared inputs for timing examples. Keep these deterministic so students see
# comparable behavior when re-running the notebook.
rand1M = [random.randint(1, 10_000_000) for _ in range(1_000_000)]
rand100000 = [random.randint(1, 1_000_000) for _ in range(100_000)]
rand10000 = [random.randint(1, 100_000) for _ in range(10_000)]
rand1000 = [random.randint(1, 100_000) for _ in range(1_000)]
rand100 = [random.randint(1, 1_000) for _ in range(100)]
rand10 = [random.randint(1, 1_000) for _ in range(10)]
rand9 = [random.randint(1, 1_000) for _ in range(9)]
rand8 = [random.randint(1, 1_000) for _ in range(8)]
rand7 = [random.randint(1, 1_000) for _ in range(7)]
rand6 = [random.randint(1, 1_000) for _ in range(6)]

In [ ]:
rand6

In [ ]:
rand6sorted = bogosort_out_of_place(rand6)
rand6, rand6sorted

In [ ]:
%%timeit
bogosort_out_of_place(rand6)

In [ ]:
rand7sorted = bogosort_out_of_place(rand7) 
rand7, rand7sorted

In [ ]:
%%timeit
bogosort_out_of_place(rand7)

In [ ]:
%%timeit
bogosort_out_of_place(rand8)

In [ ]:
# Bogosort is intentionally impractical. Try 8-10 items only if you are
# prepared to interrupt the kernel.
# bogosort(rand9)
math.factorial(9)

In [ ]:
# bogosort(rand10) would need about 10! permutations on average.
math.factorial(10)

In [ ]:
%%timeit
bogosort(rand7)

In [ ]:
rand8

In [ ]:
# bogosort(rand8)

In [ ]:
# bogosort(rand9)

In [ ]:
# %%timeit
# bogosort(rand9)

In [ ]:
rand10copy = rand10.copy()
rand10sorted = bogosort_in_place(rand10copy)
rand10copy, rand10sorted, rand10copy is rand10sorted  # in-place variant returns the same object

In [ ]:
rand6

In [ ]:
rand7

In [ ]:
# Ten random elements already means 10! possible permutations.
# bogosort_out_of_place(rand10)
math.factorial(10)

In [ ]:
math.factorial(10)

In [ ]:
math.factorial(20)  # so we are highly unlikely to sort 20 numbers with this Bogosort

In [ ]:
math.factorial(9)

In [ ]:
rand10

In [ ]:
# bogosort(rand10)

In [ ]:
bogo7 = bogosort(rand7)
bogo7

In [ ]:
# Time complexity for Bogosort is O(n * n!) on average, so it is a teaching example only.

Source: http://interactivepython.org/runestone/static/pythonds/SortSearch/toctree.html

### Bubble sort

![bubble](https://github.com/ValRCS/RTU_Algorithms_DIP321/blob/main/imgs/bubblepass.png?raw=1)

$$Complexity: O(n^2)$$

A bubble sort is often considered the most inefficient sorting method since it must exchange items before the final location is known. These “wasted” exchange operations are very costly. However, because the bubble sort makes passes through the entire unsorted portion of the list, it has the capability to do something most sorting algorithms cannot. In particular, if during a pass there are no exchanges, then we know that the list must be sorted. A bubble sort can be modified to stop early if it finds that the list has become sorted. This means that for lists that require just a few passes, a bubble sort may have an advantage in that it will recognize the sorted list and stop.

In [ ]:
l = [1,2,3,4,32,5,5,66,33,221,34,23,12]

## Bubble Sort implementation

In [ ]:
def bubblesort(seq, debug=False, return_stats=False):
    """Return a sorted copy using bubble sort.

    Worst-case and average time complexity: O(n^2).
    Best case with early stop: O(n).
    """
    nums = seq.copy()
    n = len(nums)
    total_swaps = 0

    for pass_end in range(n - 1, 0, -1):
        swaps_this_pass = 0
        for i in range(pass_end):
            if nums[i] > nums[i + 1]:
                nums[i], nums[i + 1] = nums[i + 1], nums[i]
                swaps_this_pass += 1
                total_swaps += 1
        if debug:
            print(nums, f"swaps={swaps_this_pass}")
        if swaps_this_pass == 0:
            break

    if return_stats:
        return nums, total_swaps
    return nums

In [ ]:
bubblesort(l, debug=True, return_stats=True)

In [ ]:
bubblesort(rand10, debug=True, return_stats=True)

In [ ]:
sorted_list = list(range(10))
new_list = bubblesort(sorted_list, debug=True)
new_list

In [ ]:
bubble1000 = bubblesort(rand1000)

In [ ]:
rand1000[:5]

In [ ]:
%%timeit
bubblesort(rand1000)

In [ ]:
%%timeit
bubblesort(rand10000)

In [ ]:
# so 10 times more data -> ~100x slower

In [ ]:
100_000**2

In [ ]:
# Bubble sort on 100_000 random values is deliberately skipped.
# Quadratic growth means it can take minutes or longer in Colab.
# sorted_100k = bubblesort(rand100000)

In [ ]:
# %%time
# bubblesort(rand100000)

In [ ]:
10_000**2, 100_000**2

### Selection sort

![selection](https://github.com/ValRCS/RTU_Algorithms_DIP321/blob/main/imgs/selectionsort.png?raw=1)

$$Complexity: O(n^2)$$

The selection sort improves on the bubble sort by making only one exchange for every pass through the list. In order to do this, a selection sort looks for the largest value as it makes a pass and, after completing the pass, places it in the proper location. As with a bubble sort, after the first pass, the largest item is in the correct place. After the second pass, the next largest is in place. This process continues and requires n−1 passes to sort n items, since the final item must be in place after the (n−1) st pass.

In [ ]:
l = [1,2,3,4,32,5,5,66,33,221,34,23,12]

In [ ]:
def selectionSort(seq, debug=False):
    """Return a sorted copy using selection sort.

    Time complexity is O(n^2) in best, average, and worst cases because it
    keeps scanning the unsorted prefix even if the data is already sorted.
    """
    result = seq.copy()
    n = len(result)

    for end in range(n - 1, 0, -1):
        max_idx = 0
        for i in range(1, end + 1):
            if result[i] > result[max_idx]:
                max_idx = i
        result[end], result[max_idx] = result[max_idx], result[end]
        if debug:
            print(result)

    return result

In [ ]:
selectionSort(l, debug=True)

In [ ]:
rand10

In [ ]:
selectionSort(rand10, debug=True)

The benefit of selection over bubble sort is it does one exchange per pass whereas bubble sort can do multiple exchanges.

In [ ]:
%%timeit
selectionSort(rand1000)

In [ ]:
%%timeit
selectionSort(rand10000)

In [ ]:
# so 10 x more items gives roughly 10x10 = 100x slower speed

### Insertion sort

![insertion](https://github.com/ValRCS/RTU_Algorithms_DIP321/blob/main/imgs/insertionsort.png?raw=1)

$$Complexity: O(n^2)$$

In [ ]:
l = [1,2,3,4,32,5,5,66,33,221,34,23,12]

In [ ]:
def insertionSort(seq, debug=False):
    """Return a sorted copy using insertion sort.

    Worst-case time complexity: O(n^2).
    Best case on already sorted data: O(n).
    """
    result = seq.copy()

    for i in range(1, len(result)):
        current = result[i]
        pos = i
        while pos > 0 and result[pos - 1] > current:
            result[pos] = result[pos - 1]
            pos -= 1
        result[pos] = current
        if debug:
            print(result)

    return result

In [ ]:
insertionSort(l,debug=True)

In [ ]:
rand1000 = [random.randint(1,10000) for _ in range(1_000)]
rand1000[:5]

In [ ]:
rand1000[:5]

In [ ]:
%%timeit
insertionSort(rand1000) 

In [ ]:
%%timeit
insertionSort(rand10000)

In [ ]:
rand1000[:5]

In [ ]:
%%timeit
insertionSort(rand1000)

In [ ]:
rand10000 = [random.randint(1,100000) for _ in range(10_000)]
rand10000[:5]

In [ ]:
%%timeit
insertionSort(rand10000)

In [ ]:
# so again 10 x more data -> 100x slower algorithm when complexity is quadratic O(n^2)

In [ ]:
%%timeit
insertionSort(rand1000)

In [ ]:
rand20k = [random.randint(1,1000000) for _ in range(20_000)]
rand20k[:10]

In [ ]:
%%timeit
insertionSort(rand20k)

In [ ]:
rand20k[:10]

In [ ]:
%%time
sorted20k = insertionSort(rand20k)

In [ ]:
sorted20k[:5], rand20k[:5]

In [ ]:
sorted20k[9000] = 777
sorted20k[12000] = 555
# so not sorted anymore

### Insertion Sort - Conclusion

Insertion sort is still $O(n^2)$ in the worst case, but it is often faster than bubble sort and selection sort for small or nearly sorted inputs. It performs few writes when the input is close to sorted, which is one reason it is used inside practical hybrid sorting algorithms.

## Insertion Sort on Partially Sorted Data

Insertion sort should beat selection sort and bubble sort on partially sorted inputs because many elements are already close to their final positions.

In [ ]:
partially_sorted_1k = list(range(1_000))
for _ in range(100):
    partially_sorted_1k[random.randint(0, 999)] = random.randint(-100_000, 100_000)

partially_sorted_1k[:20]

In [ ]:
# Compare all four approaches on the same mostly sorted data.
# %%timeit
# bubblesort(partially_sorted_1k)

In [ ]:
# %%timeit
# selectionSort(partially_sorted_1k)

In [ ]:
# %%timeit
# insertionSort(partially_sorted_1k)

In [ ]:
# %%timeit
# sorted(partially_sorted_1k)

### Merge Sort

In [ ]:
# first implementation idea by https://en.wikipedia.org/wiki/John_von_Neumann


![merge](https://github.com/ValRCS/RTU_Algorithms_DIP321/blob/main/imgs/mergesort.png?raw=1)

![merge1](https://github.com/ValRCS/RTU_Algorithms_DIP321/blob/main/imgs/mergesortB.png?raw=1)

$$Complexity: O(nlog(n))$$

In [ ]:
l = [1,2,3,4,32,5,5,66,33,221,34,23,12]

In [ ]:
def merge(left, right):
    """Merge two sorted lists into one sorted list."""
    result = []
    i = 0
    j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:  # <= keeps merge sort stable
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    result.extend(left[i:])
    result.extend(right[j:])
    return result


def mergeSort(seq, debug=False):
    """Return a sorted copy using merge sort."""
    if len(seq) <= 1:
        return seq.copy()

    mid = len(seq) // 2
    left = mergeSort(seq[:mid], debug=debug)
    right = mergeSort(seq[mid:], debug=debug)
    result = merge(left, right)

    if debug:
        print("Merging", result)
    return result

In [ ]:
mergeSort(l, debug=True)

In [ ]:
l

In [ ]:
sorted_list = mergeSort(l)
sorted_list

In [ ]:
rand1000[:10]

In [ ]:
shuffle(rand1000)

In [ ]:
rand1000[:10]

In [ ]:
merge1000 = mergeSort(rand1000)
merge1000[:20]

In [ ]:
rand1000[:10]

In [ ]:
merge10k = mergeSort(rand10000)
merge10k[:10]

In [ ]:
%%timeit
mergeSort(rand1000)

In [ ]:
%%timeit
mergeSort(rand10000)

In [ ]:
r100_000 = [random.randint(1,1_000_000) for _ in range(100_000)]
r100_000[:5]

In [ ]:
%%timeit
mergeSort(rand100000)  # n log n starts to shine here

In [ ]:
shuffle(r100_000)

In [ ]:
r100_000[:5]

In [ ]:
# Python has a built in sorted - as most languages do, lets test it

In [ ]:
%%timeit
sorted(r100_000) # sorted uses timsort which is a combination of insertion sort + merge sort

In [ ]:
%%timeit
sorted(rand10000)

In [ ]:
r100_000[:5]

In [ ]:
shuffle(r100_000)

In [ ]:
r100_000[:5]

In [ ]:
%%timeit
sorted(r100_000) # sorted uses timsort which is a combination of insertion sort + merge sort

In [ ]:
sorted100k = mergeSort(r100_000)
sorted100k[:5]

In [ ]:
%%timeit
mergeSort(sorted100k) # so we gained nothing from being sorted

In [ ]:
%%timeit
insertionSort(sorted100k)  # insertion sort should be slow we shall see


In [ ]:
sorted100k[50000:50000+10]

In [ ]:
sorted100k[555] = 400000
sorted100k[9000] = 1333


In [ ]:
%%timeit
insertionSort(sorted100k) 

## Hybrid Sorts

Modern library sorts are hybrids because real data is not always random. Python's `sorted` and `list.sort` use Timsort, a stable adaptive algorithm derived from merge sort and insertion sort ideas.

Timsort detects already sorted runs, extends short runs efficiently, and merges runs in a way that keeps worst-case performance at $O(n \log n)$. It can be close to $O(n)$ on already sorted or mostly sorted data.

# Big O, Theta and Omega bounds

### Merge sort recurrence

Merge sort is described by the recurrence:

$$T(n) = 2T(n/2) + n$$

Each call sorts two halves, and the merge step performs linear work.

In [ ]:
# so how to prove that merge sort is really O(n log n) time complexity?
# in reality we are looking for tight bound the Θ(n log n) complexity
# so O is very loose, in every day usage when people say O they really mean Θ - theta
# O is showing that the algorith is no worse than some f(n)
# I could say that merge sort is O(n!) and it would still be correct but practically useless
# since most algorithms are O(n!)
# so Merge sort is O(n!), O(n^5),O(n^2) and so on and finally most crucially O(n log n)
# Merge sort is NOT O(n)
# so thats what Θ(n log n)

In [ ]:
## So ideas on how to prove mergesort is O(n log n) ?
## Instinctively we see that we are dividing in halves and solving the problem for those

# There is something called Master Theorem which lets us quickly see the solution for most types of recurrence

In [ ]:
# what is a recurrence relation then?

A recurrence relation defines a recursive runtime in terms of smaller inputs. For merge sort, $T(n) = 2T(n/2) + n$ solves to $\Theta(n \log n)$ by the Master Theorem.

In [ ]:
# Next we use the Master Theorem to solve recurrences such as merge sort's T(n) = 2T(n/2) + n.

In [ ]:
# Recurrences are most useful for recursive algorithms. For simple loops, we usually count loop iterations directly.

### Quick sort

![quick](https://github.com/ValRCS/RTU_Algorithms_DIP321/blob/main/imgs/quicksort.png?raw=1)

$$Complexity: O(nlog(n))$$ $$Worst case : O(n^2)$$

In [ ]:
def quickSort(seq):
    """Return a sorted copy using textbook quicksort partitioning.

    Average time complexity: O(n log n).
    Worst case with unlucky pivots: O(n^2).
    """
    result = seq.copy()
    quickSortHelper(result, 0, len(result) - 1)
    return result


def quickSortHelper(alist, first, last):
    if first < last:
        splitpoint = partition(alist, first, last)
        quickSortHelper(alist, first, splitpoint - 1)
        quickSortHelper(alist, splitpoint + 1, last)


def partition(alist, first, last):
    pivotvalue = alist[first]
    leftmark = first + 1
    rightmark = last

    while True:
        while leftmark <= rightmark and alist[leftmark] <= pivotvalue:
            leftmark += 1
        while leftmark <= rightmark and alist[rightmark] >= pivotvalue:
            rightmark -= 1

        if rightmark < leftmark:
            break
        alist[leftmark], alist[rightmark] = alist[rightmark], alist[leftmark]

    alist[first], alist[rightmark] = alist[rightmark], alist[first]
    return rightmark


alist = [54, 26, 93, 17, 77, 31, 44, 55, 20]
sorted_list = quickSort(alist)
alist, sorted_list  # original preserved, sorted copy returned

In [ ]:
sorted_list

In [ ]:
%%timeit
quickSort(rand1000)

In [ ]:
%%timeit
quickSort(rand10000)

In [ ]:
%%timeit
quickSort(rand100000)

In [ ]:
%%timeit
quickSort(rand1M)

In [ ]:
%%timeit
sorted(rand1M)

In [ ]:
%%timeit
sorted(rand100000)

# Heap Sort

In [ ]:
# Idea for heap sort is to use selection sort as a basis
# but we use a data structure such as heap to get min (or max) values out of the heap

In [ ]:
import heapq

In [ ]:
# https://docs.python.org/3/library/heapq.html
def heapsort(iterable):
    """Return a sorted list using a binary heap."""
    heap = []
    for value in iterable:
        heapq.heappush(heap, value)  # O(log n)
    return [heapq.heappop(heap) for _ in range(len(heap))]  # n pops, each O(log n)

In [ ]:
rand9

In [ ]:
heapsort(rand9)

In [ ]:
%%timeit
heapsort(rand100000)

In [ ]:
%%timeit
heapsort(rand1M)

In [ ]:
# Stable sorts and unstable sorts

In [ ]:
my_tuples = [(c, random.randint(1,10)) for c in "Riga Technical University"]
my_tuples

In [ ]:
sorted(my_tuples) # this sorts by the first part of tuple: the alphabetical, lexicographical key - ASCII codes
# https://www.asciitable.com/

In [ ]:
sorted(my_tuples, key=lambda element: element[1])  # sort by the second tuple item

In [ ]:
# Python's Timsort is stable: items with equal keys keep their original relative order.
# https://en.wikipedia.org/wiki/Timsort

# Counting Sort
Can we do better than O(n log n) time?

In [ ]:
# if we have specific data we can
# one place where counting sort could be useful could be if we have to sort many numbers over a limited range


In [ ]:
def countingSort(seq, min_value=None, max_value=None):
    """Return a sorted copy using counting sort for bounded integer data.

    Time complexity: O(n + k), where k is the value range.
    Space complexity: O(k). This is efficient only when k is reasonably small.
    """
    if not seq:
        return []

    if min_value is None:
        min_value = min(seq)
    if max_value is None:
        max_value = max(seq)
    if min_value > max_value:
        raise ValueError("min_value must be <= max_value")

    buckets = [0] * (max_value - min_value + 1)
    for value in seq:
        if value < min_value or value > max_value:
            raise ValueError(f"value {value} is outside the configured range")
        buckets[value - min_value] += 1

    result = []
    for offset, count in enumerate(buckets):
        result.extend([offset + min_value] * count)
    return result

In [ ]:
dice = [random.randint(1,6) for _ in  range(20)]
dice

In [ ]:
sorted(dice)

In [ ]:
countingSort(dice)

In [ ]:
magic_dice_100k = [random.randint(1,20) for _ in  range(100_000)]
magic_dice_100k[:5]

In [ ]:
%%timeit
sorted(magic_dice_100k)

In [ ]:
sorted_magic = countingSort(magic_dice_100k, min_value=1, max_value=20)
sorted_magic[:5]

In [ ]:
%%timeit
countingSort(magic_dice_100k, min_value=1, max_value=20)

In [ ]:
magic_dice_1M = [random.randint(1,20) for _ in  range(1_000_000)]
magic_dice_1M[:5]

In [ ]:
min(magic_dice_1M),max(magic_dice_1M)

In [ ]:
%%timeit
sorted(magic_dice_1M)

In [ ]:
%%timeit
countingSort(magic_dice_1M, min_value=1, max_value=20)

In [ ]:
# Counting sort needs a bounded integer range ahead of time.
# It is inefficient if the range is huge and sparse, but excellent for many
# repeated values from a compact range.

In [ ]:
# Related non-comparison sorts:
# https://en.wikipedia.org/wiki/Bucket_sort
# https://en.wikipedia.org/wiki/Radix_sort
# https://en.wikipedia.org/wiki/Counting_sort

# # Quad Sort new sort ?


In [ ]:
# Quad Sort: https://github.com/scandum/quadsort

# References and useful links

- Sorting visualizations: https://visualgo.net/en/sorting
- Big O notation: https://runestone.academy/ns/books/published/pythonds/AlgorithmAnalysis/BigONotation.html
- Python sorting HOWTO: https://docs.python.org/3/howto/sorting.html
- Python `heapq`: https://docs.python.org/3/library/heapq.html